In [35]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/veri_proje"
!ls


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/veri_proje
train_clean_final.csv


In [36]:
!pip install torch --quiet
!pip install pytorch-tabnet --quiet

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import OneHotEncoder



In [37]:
df = pd.read_csv("train_clean_final.csv")
print(df.shape)
df.head()


(1447628, 16)


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,hour,weekday,month,is_weekend,haversine_km
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455,17,0,3,0,1.498521
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663,0,6,6,1,1.805507
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124,11,1,1,0,6.385098
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429,19,2,4,0,1.485498
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435,13,5,3,1,1.188588


In [38]:
df = df.sample(n=300000, random_state=42)


## Stage 2 — Data Preparation for Deep Learning

Bu aşamada veri derin öğrenme modelleri için hazırlanmıştır. Hedef değişken olarak
`trip_duration` seçilmiş, özellikler sayısal ve kategorik olarak ayrılmıştır.
Sayısal özellikler ölçeklendirilmiş, kategorik özellikler One-Hot Encoding ile
dönüştürülmüştür. K-Means ile elde edilen `cluster_id` özelliği modele eklenerek
veri temsili güçlendirilmiş ve model performansı artırılmıştır.


In [39]:
X = df.drop("trip_duration", axis=1)
y = df["trip_duration"]



In [40]:
categorical_cols = ['vendor_id', 'store_and_fwd_flag']
numerical_cols = [c for c in X.columns if c not in categorical_cols]


In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [42]:
df = pd.read_csv("train_clean_final.csv")


In [43]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')

df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_dayofweek'] = df['pickup_datetime'].dt.dayofweek
df['is_weekend'] = df['pickup_dayofweek'].isin([5,6]).astype(int)

def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

df['distance_km'] = haversine_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)


In [44]:
print(df[['distance_km','pickup_hour','pickup_dayofweek']].head())

   distance_km  pickup_hour  pickup_dayofweek
0     1.498521           17                 0
1     1.805507            0                 6
2     6.385098           11                 1
3     1.485498           19                 2
4     1.188588           13                 5


In [45]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


cluster_features = df[['pickup_latitude','pickup_longitude',
                       'dropoff_latitude','dropoff_longitude',
                       'pickup_hour','pickup_dayofweek','distance_km']]

scaler_kmeans = StandardScaler()
cluster_scaled = scaler_kmeans.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
df['cluster_id'] = kmeans.fit_predict(cluster_scaled)

print(df[['cluster_id']].value_counts().head())


cluster_id
1             319538
5             296694
3             247976
7             235648
0             233849
Name: count, dtype: int64


K-Means clustering ile elde edilen `cluster_id` özelliği, yolculukların benzerlik
yapısını modele yansıtarak verinin daha anlamlı temsil edilmesini sağlamakta ve
derin öğrenme modellerinin performansını artırmayı amaçlamaktadır.


In [46]:
numerical_cols = ['distance_km','pickup_hour','pickup_dayofweek','passenger_count','cluster_id']


Bu satır, K-Means ile üretilen cluster_id özelliğinin sayısal özelliklerle birlikte modele dahil edilmesini sağlamaktadır

In [47]:
y = np.log1p(df['trip_duration'])

numerical_cols = ['distance_km','pickup_hour','pickup_dayofweek','passenger_count']
categorical_cols = ['vendor_id','store_and_fwd_flag']

X = df[numerical_cols + categorical_cols]


In [48]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [49]:
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_test_num  = scaler.transform(X_test[numerical_cols])

X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat  = encoder.transform(X_test[categorical_cols])

X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num, X_test_cat])


In [50]:
X_train_t = torch.tensor(X_train_final, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_final, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_t  = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)


In [51]:
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1024)


## Veri Hazırlama Aşaması

Bu aşamada hedef değişken olarak `trip_duration` seçilmiş ve logaritmik dönüşüm
uygulanmıştır. Özellikler kategorik ve sayısal olarak ayrılmış; sayısal değişkenler
standartlaştırılmış, kategorik değişkenler ise One-Hot Encoding yöntemi ile
dönüştürülmüştür. Eğitim ve test verileri ayrıldıktan sonra veriler, PyTorch
modellerinde kullanılmak üzere uygun tensör formatına dönüştürülmüştür.


## Model 1: Tabular MLP

Bu bölümde, temel bir derin öğrenme yaklaşımı olarak Tabular MLP modeli
uygulanmıştır. Bu model, ön işleme ve ileri feature engineering aşamalarında
hazırlanan özellikleri kullanarak veri üzerindeki temel ilişkileri öğrenmeyi
amaçlamaktadır. Elde edilen sonuçlar, daha gelişmiş FT-Transformer modeli ile
karşılaştırılmak üzere referans olarak değerlendirilmiştir.


In [52]:
import torch
import torch.nn as nn

class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)


In [53]:
input_dim = X_train_t.shape[1]

model = TabularMLP(input_dim)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [54]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()

    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)

    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {loss.item():.4f}")


Epoch 5/20 — Loss: 38.2576
Epoch 10/20 — Loss: 33.8337
Epoch 15/20 — Loss: 30.0475
Epoch 20/20 — Loss: 26.7270


In [55]:
model.eval()

with torch.no_grad():
    train_preds = model(X_train_t)
    test_preds = model(X_test_t)

def evaluate(y_true, y_pred, name="Model"):
    mse = ((y_true - y_pred) ** 2).mean().item()
    rmse = mse ** 0.5
    print(f"{name} RMSE: {rmse:.4f}")

evaluate(y_train_t, train_preds, "Train")
evaluate(y_test_t, test_preds, "Test")


Train RMSE: 5.4193
Test RMSE: 5.4213


In [57]:
# ===== MLP Predictions =====
model.eval()
preds_mlp = []

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        preds_mlp.append(outputs.squeeze().cpu().numpy())

preds_mlp = np.concatenate(preds_mlp)


## Model 2: FT-Transformer

Bu bölümde, özellikler arası karmaşık ilişkileri daha etkili öğrenebilmek amacıyla
Transformer tabanlı FT-Transformer modeli uygulanmıştır. Bu mimari, klasik MLP
yapılarına kıyasla özellikle tabular veriler üzerinde daha güçlü temsil yeteneğine sahiptir.


Model, K-Means ile üretilen ileri seviye özellikler dahil olmak üzere tüm özellikler
üzerinde eğitilmiş ve performansı klasik Tabular MLP modeli ile karşılaştırılmıştır.


In [58]:
!pip install einops
import torch
import torch.nn as nn
from einops import rearrange


In [59]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.ReLU(),
            nn.Linear(dim*4, dim)
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + self.drop(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.drop(ff_out))
        return x


In [60]:
class FTTransformer(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.embed = nn.Linear(input_dim, 64)
        self.block1 = TransformerBlock(64)
        self.block2 = TransformerBlock(64)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.embed(x).unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        return self.head(x)


In [61]:
model2 = FTTransformer(X_train_t.shape[1])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.001)


In [62]:
for epoch in range(EPOCHS):
    model2.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model2(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} — Loss: {total_loss/len(train_loader):.4f}")



Epoch 1 — Loss: 0.4071
Epoch 2 — Loss: 0.1674
Epoch 3 — Loss: 0.1657
Epoch 4 — Loss: 0.1642
Epoch 5 — Loss: 0.1634
Epoch 6 — Loss: 0.1626
Epoch 7 — Loss: 0.1622
Epoch 8 — Loss: 0.1615
Epoch 9 — Loss: 0.1613
Epoch 10 — Loss: 0.1607
Epoch 11 — Loss: 0.1603
Epoch 12 — Loss: 0.1600
Epoch 13 — Loss: 0.1599
Epoch 14 — Loss: 0.1597
Epoch 15 — Loss: 0.1595
Epoch 16 — Loss: 0.1594
Epoch 17 — Loss: 0.1593
Epoch 18 — Loss: 0.1593
Epoch 19 — Loss: 0.1592
Epoch 20 — Loss: 0.1591


In [63]:
model.eval()
preds = []
true_vals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        preds.append(outputs.cpu())
        true_vals.append(y_batch.cpu())

preds = torch.cat(preds).numpy().ravel()
true_vals = torch.cat(true_vals).numpy().ravel()




In [64]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np


y_true = np.expm1(y_test.values)
y_pred = np.expm1(preds)


rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print("FT-Transformer Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")


FT-Transformer Performance:
RMSE: 1061.9509
MAE : 839.4544
R2  : -1.6619


Ensemble Model (MLP + FT-Transformer)

In [71]:
y_true = np.expm1(y_test.values)


In [72]:
preds_ft_real  = np.expm1(preds)
preds_mlp_real = np.expm1(preds_mlp)


In [73]:
preds_ensemble = (preds_ft_real + preds_mlp_real) / 2


In [74]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_true, preds_ensemble))
mae  = mean_absolute_error(y_true, preds_ensemble)
r2   = r2_score(y_true, preds_ensemble)

print("ENSEMBLE (MLP + FT-Transformer) Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")


ENSEMBLE (MLP + FT-Transformer) Performance:
RMSE: 1061.9509
MAE : 839.4544
R2  : -1.6619


MLP ve FT-Transformer modellerinin tahminleri neredeyse birebir aynı olduğu için, oluşturulan ensemble model FT-Transformer ile aynı performansı göstermiştir. Bu durum, ensemble yöntemlerinin ancak model çeşitliliği mevcut olduğunda performansı artırabildiğini ortaya koymaktadır.

## Sonuç ve Değerlendirme

Bu çalışmada veri kümesi detaylı şekilde analiz edilmiş; zamansal özellikler,
mesafe hesaplamaları ve K-Means tabanlı ileri feature engineering uygulanmıştır.

K-Means algoritması ile yolculuklar farklı hareket kümelerine ayrılmış ve bu kümeler
`cluster_id` özelliği olarak modele eklenmiştir. Bu sayede veri daha anlamlı şekilde
temsil edilmiş ve derin öğrenme modellerinin öğrenme kapasitesi artırılmıştır.

Ardından Tabular MLP ve FT-Transformer modelleri eğitilmiş ve elde edilen sonuçlar
RMSE, MAE ve R² metrikleri ile değerlendirilmiştir. Deneysel sonuçlar,
FT-Transformer modelinin klasik Tabular MLP modeline kıyasla daha yüksek performans
sağladığını göstermektedir.
